**Interaktive Exploration der Proximity-Funktion**
In der unterliegenen Zelle können die Parameter der Proximity-Funktion (z. B. Schärfe, Frequenzabfall, maximal zu berücksichtigende Frequenz) über ein interaktives Widget dynamisch verändert und die resultierende Kurve visualisiert werden. Dies erlaubt eine intuitive Erkundung der Funktion und ihrer Auswirkungen auf die Bewertung der rhythmischen Verhältnisse.

In [ ]:
# Proximity Funktion mit Cos-Peaks
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
import numpy as np
from scipy.integrate import simpson
from scipy.stats import gaussian_kde
import ipywidgets as widgets
from ipywidgets import interactive_output

# Proximity-Funktion: bewertet Zahlenverhältnisse nach harmonischen Kriterien
def proximity_max_freq(
    x,
    sharpness_base=10.0,
    sharpness_growth=2.0,
    decay=0.1,
    max_freq=4,
    weight_exponent=1.0,
    freq_sharpness_exp=1.0,
    norm_x=1.0,
    output_max=1.0
):
    x = np.array(x)
    all_freq_values = []

    for freq in range(1, max_freq + 1):
        weight = 1 / (freq ** weight_exponent)
        freq_sharpness_base = sharpness_base * (freq ** freq_sharpness_exp)
        freq_sharpness_growth = sharpness_growth * (freq ** freq_sharpness_exp)
        sharpness = freq_sharpness_base * (x ** freq_sharpness_growth)
        cos_term = ((1 + np.cos(2 * np.pi * freq * x)) / 2) ** sharpness

        # Normierung auf norm_x
        peak_sharpness = freq_sharpness_base * (norm_x ** freq_sharpness_growth)
        peak_value = ((1 + np.cos(2 * np.pi * freq * norm_x)) / 2) ** peak_sharpness
        cos_term_normalized = cos_term / peak_value

        val = weight * cos_term_normalized / (x ** decay)
        all_freq_values.append(val)

    result = np.maximum.reduce(all_freq_values)

    max_at_norm_x = np.max([
        (1 / (freq ** weight_exponent)) * 1.0 / (norm_x ** decay)
        for freq in range(1, max_freq + 1)
    ])
    return result / max_at_norm_x * output_max

# Plot-Funktion für Proximity und dessen Verteilung
def plot_proximity_max_freq(
    sharpness_base=10.0,
    sharpness_growth=2.0,
    decay=0.5,
    max_freq=6,
    weight_exponent=1.0,
    freq_sharpness_exp=0.5
):
    x_vals = np.linspace(1, 5, 1000)
    y_vals = proximity_max_freq(
        x_vals,
        sharpness_base=sharpness_base,
        sharpness_growth=sharpness_growth,
        decay=decay,
        max_freq=max_freq,
        weight_exponent=weight_exponent,
        freq_sharpness_exp=freq_sharpness_exp,
        output_max=1.0,
    )

    # Integral = numerischer Mittelwert der Funktion
    mean_prox = simpson(y_vals, x=x_vals) / (x_vals[-1] - x_vals[0])

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle("Einfluss von Parametern auf Proximity-Funktion", fontsize=16, y=1.02)

    # Plot der Proximity-Funktion
    axes[0].plot(x_vals, y_vals, label=f"Proximity (∫mean ≈ {mean_prox:.3f})")
    axes[0].plot(
        x_vals,
        [mean_prox] * len(x_vals),
        color='red',
        linestyle='--',
        linewidth=2,
        label=f"∫mean ≈ {mean_prox:.3f}"
    )
    axes[0].set_xlabel("x (Verhältnis)")
    axes[0].set_ylabel("Proximity")
    axes[0].set_title("Proximity mit frequenzabhängiger Sharpness")
    axes[0].set_ylim(0, 1.05)
    axes[0].grid(True, linestyle="--", alpha=0.3)
    axes[0].legend()

    # Histogramm + KDE (Verteilung von y_vals)
    counts, bins_hist, _ = axes[1].hist(
        y_vals,
        bins=50,
        range=(0, 1),
        alpha=0.4,
        color="skyblue",
        density=True,
        label="Histogramm"
    )

    kde = gaussian_kde(y_vals)
    x_dens = np.linspace(0, 1, 5000)
    kde_vals = kde(x_dens)

    # Automatische Skalierung
    scaling_factor = max(counts) / max(kde_vals)
    axes[1].plot(
        x_dens,
        kde_vals * scaling_factor,
        color="darkblue",
        lw=2,
        label="Dichte (KDE)"
    )

    axes[1].axvline(
        mean_prox,
        color='red',
        linestyle='--',
        linewidth=2,
        label=f"∫mean ≈ {mean_prox:.3f}"
    )
    axes[1].set_title("Proximity-Verteilung")
    axes[1].set_xlabel("Proximity")
    axes[1].set_ylabel("Normierte Häufigkeit")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.3)

    plt.tight_layout()
    plt.subplots_adjust(top=0.90)
    plt.show()

# Interaktive Visualisierung
interact(
    plot_proximity_max_freq,
    sharpness_base=FloatSlider(min=1, max=50, step=1, value=10, description="Sharpness Base"),
    sharpness_growth=FloatSlider(min=0, max=3, step=0.1, value=2, description="Sharpness Growth"),
    decay=FloatSlider(min=0, max=3, step=0.05, value=0.1, description="Decay"),
    max_freq=IntSlider(min=1, max=12, step=1, value=4, description="Max Frequency"),
    weight_exponent=FloatSlider(min=0, max=3, step=0.1, value=1, description="Weight Exponent"),
    freq_sharpness_exp=FloatSlider(min=0, max=3, step=0.1, value=1, description="Freq Sharpness Exp"),
)


interactive(children=(FloatSlider(value=10.0, description='Sharpness Base', max=50.0, min=1.0, step=1.0), Floa…

<function __main__.plot_proximity_max_freq(sharpness_base=10.0, sharpness_growth=2.0, decay=0.5, max_freq=6, weight_exponent=1.0, freq_sharpness_exp=0.5)>

In [ ]:
# Proximity Funktion mit Gauss-Peaks
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
import numpy as np
from scipy.integrate import simpson
from scipy.stats import gaussian_kde

def proximity_max_freq_gaussian(
    x,
    sharpness_base=10.0,
    sharpness_growth=1.0,   # steuert Breite mit k
    decay=0.1,              # exponentielles Abklingen mit k (ab k=2)
    max_freq=4,
    weight_exponent=1.0,    # f^-w
    freq_sharpness_exp=1.0, # steuert Breite mit f
    norm_x=1.0,
    output_max=1.0
):
    x = np.array(x)
    all_freq_values = []

    for f in range(1, max_freq + 1):
        # Frequenzgewichtung
        weight = 1 / (f ** weight_exponent)

        gauss_max = np.zeros_like(x, dtype=float)
        max_k = int(np.ceil(x.max() * f)) + 1

        for k in range(1, max_k + 1):
            mu = k / f

            # Sharpness abhängig von f & k
            sharpness = sharpness_base * (f ** freq_sharpness_exp) * (k ** sharpness_growth)
            sigma = 1 / (sharpness + 1e-9)

            # k-Dämpfung: bei k=1 = 1
            mu_decay = 1 / (mu ** decay)

            # Gaussian-Peak
            g = np.exp(-0.5 * ((x - mu) / sigma) ** 2)

            gauss_max = np.maximum(gauss_max, weight * mu_decay * g)

        all_freq_values.append(gauss_max)

    result = np.maximum.reduce(all_freq_values)

    # Normierung auf output_max bei norm_x
    max_at_norm_x = np.max([
        (1 / (f ** weight_exponent))
        for f in range(1, max_freq + 1)
    ])
    return result / (max_at_norm_x + 1e-9) * output_max

def plot_proximity_max_freq_gaussian(
    sharpness_base=10.0,
    sharpness_growth=1.0,
    decay=0.5,
    max_freq=6,
    weight_exponent=1.0,
    freq_sharpness_exp=0.5
):
    x_vals = np.linspace(1, 5, 2000)
    y_vals = proximity_max_freq_gaussian(
        x_vals,
        sharpness_base=sharpness_base,
        sharpness_growth=sharpness_growth,
        decay=decay,
        max_freq=max_freq,
        weight_exponent=weight_exponent,
        freq_sharpness_exp=freq_sharpness_exp,
        output_max=1.0,
    )

    mean_prox = simpson(y_vals, x=x_vals) / (x_vals[-1] - x_vals[0])

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle("Proximity-Funktion mit k- und f-abhängigen Gauß-Peaks", fontsize=16, y=1.02)

    # Funktion plotten
    axes[0].plot(x_vals, y_vals, label=f"Proximity (∫mean ≈ {mean_prox:.3f})")
    axes[0].axhline(mean_prox, color='red', linestyle='--', linewidth=2,
                    label=f"∫mean ≈ {mean_prox:.3f}")
    axes[0].set_xlabel("x (Verhältnis)")
    axes[0].set_ylabel("Proximity")
    axes[0].set_title("Proximity mit k-Decay & f-Weight")
    axes[0].set_ylim(0, 1.05)
    axes[0].grid(True, linestyle="--", alpha=0.3)
    axes[0].legend()

    # Histogramm + KDE
    counts, _, _ = axes[1].hist(
        y_vals, bins=50, range=(0, 1),
        alpha=0.4, color="skyblue", density=True, label="Histogramm"
    )
    kde = gaussian_kde(y_vals)
    x_dens = np.linspace(0, 1, 5000)
    kde_vals = kde(x_dens)

    scaling_factor = max(counts) / max(kde_vals)
    axes[1].plot(x_dens, kde_vals * scaling_factor,
                 color="darkblue", lw=2, label="Dichte (KDE)")
    axes[1].axvline(mean_prox, color='red', linestyle='--',
                    linewidth=2, label=f"∫mean ≈ {mean_prox:.3f}")
    axes[1].set_title("Proximity-Verteilung")
    axes[1].set_xlabel("Proximity")
    axes[1].set_ylabel("Normierte Häufigkeit")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.3)

    plt.tight_layout()
    plt.subplots_adjust(top=0.90)
    plt.show()

# Interaktive Kontrolle
interact(
    plot_proximity_max_freq_gaussian,
    sharpness_base=FloatSlider(min=1, max=50, step=1, value=10, description="Sharpness Base"),
    sharpness_growth=FloatSlider(min=0, max=3, step=0.1, value=1, description="Sharpness Growth (k)"),
    decay=FloatSlider(min=0, max=3, step=0.05, value=0.5, description="Decay (mu)"),
    max_freq=IntSlider(min=1, max=6, step=1, value=4, description="Max Frequency"),
    weight_exponent=FloatSlider(min=0, max=3, step=0.1, value=1, description="Weight Exponent (f)"),
    freq_sharpness_exp=FloatSlider(min=0, max=3, step=0.1, value=1, description="Freq Sharpness Exp (f)"),
)

interactive(children=(FloatSlider(value=10.0, description='Sharpness Base', max=50.0, min=1.0, step=1.0), Floa…

<function __main__.plot_proximity_max_freq_gaussian(sharpness_base=10.0, sharpness_growth=1.0, decay=0.5, max_freq=6, weight_exponent=1.0, freq_sharpness_exp=0.5)>

In [ ]:
# Proximity function based on the Delta-method normal approximation (Weber-based sigma)
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from math import ceil

def proximity_weber_gauss(
    x,
    s=0.025,                 # Weber fraction (CV)
    decay=0.5,               # mu decay exponent: mu_decay = 1 / mu**decay
    max_freq=6,              # maximum f
    weight_exponent=1.0,     # weight = 1 / f**weight_exponent
    output_max=1.0,
    eps=1e-12
):
    """
    Proximity based on approximate ratio-distribution sigma:
      sigma(mu) = sqrt(2) * s * mu
      mu = k / f
      mu_decay = 1 / mu**decay
      weight = 1 / f**weight_exponent
    x can be scalar or numpy array (ratios >= 1)
    Returns array of same shape as x with proximity scores in [0, output_max]
    """
    scalar_input = np.isscalar(x)
    x = np.atleast_1d(np.array(x, dtype=float))
    xmax = float(np.nanmax(x))
    all_vals = []
    
    for f in range(1, int(max_freq) + 1):
        weight = 1.0 / (f ** weight_exponent)
        gauss_max = np.zeros_like(x, dtype=float)
        max_k = int(ceil(xmax * f)) + 2
        for k in range(1, max_k + 1):
            mu = k / f
            sigma = np.sqrt(2.0) * s * mu + eps
            mu_decay = 1.0 / (mu ** decay)
            # gaussian evaluated for all x
            g = np.exp(-0.5 * ((x - mu) / sigma) ** 2)
            val = weight * mu_decay * g
            gauss_max = np.maximum(gauss_max, val)
        all_vals.append(gauss_max)
    
    result = np.maximum.reduce(all_vals)
    # normalization: maximum at f=1, mu=1 gives weight=1, mu_decay=1 -> max_at_norm = 1
    max_at_norm = 1.0
    result = result / (max_at_norm + eps) * output_max
    return result[0] if scalar_input else result

# Quick demonstration: plot over x and show scores for typical ratios
x_grid = np.linspace(1.0, 8.0, 1401)
params = {"s": 0.025, "decay": 0.5, "max_freq": 6, "weight_exponent": 1.0}
y = proximity_weber_gauss(x_grid, **params)

# Plot
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(x_grid, y, lw=1.5)
ax.set_xlim(1,8)
ax.set_ylim(0,1.05)
ax.set_xlabel("Ratio x (IOI_long / IOI_short)")
ax.set_ylabel("Proximity")
ax.set_title(f"Proximity (Weber σ) — s={params['s']}, decay={params['decay']}, max_freq={params['max_freq']}, w_exp={params['weight_exponent']}")
ax.grid(True, linestyle="--", alpha=0.3)
plt.show()

# Scores for selected ratios
test_ratios = np.array([1.0, 1.25, 4/3, 1.49, 1.5, 1.618, 2.0, 4.0, 8.0])
scores = proximity_weber_gauss(test_ratios, **params)
df_scores = pd.DataFrame({"ratio": test_ratios, "proximity": scores})
display(df_scores)

# Save CSV for inspection
df_scores.to_csv("/mnt/data/proximity_weber_scores.csv", index=False)
print("Saved /mnt/data/proximity_weber_scores.csv")
